# 05 · Near-copy pairs inside validation and test

The hard part of the task is human vs fusion, and a fusion text is an edited copy of one human text. A human text and
its edited versions are always on the same side of the train/test split, so a fusion text in test should usually
have its human original in test as well. This notebook checks whether that structure is really there before any
model is built on it:

1. For every text, find the most similar *other* text of the same domain and language **in the same set**
   (validation or test). TF-IDF vocabularies are fitted on training texts only.
2. Compare how often texts have a near copy (cosine ≥ 0.4, the linking threshold of notebook 01) in validation and
   in test. If the shares are similar, test has the same structure as validation.
3. In validation, where labels are known: which labels do near-copy partners have, and in a human–fusion pair, how
   often does the saved XLM-R model give the higher human probability to the right text? Choosing the original out
   of two texts may be much easier than labelling one text alone.

Nothing here is used for a submission yet.

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
P = env.setup(seed=42)
ensure_data(P)

In [ ]:
# Imports; training part, validation and test texts with domain + language
import gc
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from hwkit.split import LABELS, load_split

NOTEBOOK_START = time.time()
LINK = 0.4                      # near-copy threshold, the same as for the source groups in notebook 01
FIT_PER_DOMAIN = 20_000         # training texts per domain used to fit the vocabulary (enough for the IDF weights)

train = load_split(pd.read_csv(P.data / "train.csv"))
test = pd.read_csv(P.data / "test.csv")
for df in (train, test):
    df["domain_lang"] = df["domain"] + " | " + df["lang"]
val = train[train["fold"] == 0].reset_index(drop=True)
fit_part = train[train["fold"] != 0]
del train
gc.collect()
print(f"validation: {len(val)}, test: {len(test)}")

In [ ]:
# For every validation and test text: most similar other text of the same domain and language in the same set
def nearest_other(X, chunk=2000):
    """Best cosine similarity of each row to any other row of X, and the index of that row."""
    best, arg = [], []
    for i in range(0, X.shape[0], chunk):
        S = (X[i:i + chunk] @ X.T).toarray()
        rows = np.arange(S.shape[0])
        S[rows, rows + i] = -1                          # a text must not match itself
        a = S.argmax(axis=1)
        best.append(S[rows, a])
        arg.append(a)
    return np.concatenate(best), np.concatenate(arg)

t0 = time.time()
for df in (val, test):
    df["nn_sim"], df["nn_pos"] = 0.0, -1
for dl, fit_d in fit_part.groupby("domain_lang"):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, max_features=300_000,
                          sublinear_tf=True, dtype=np.float32)
    vec.fit(fit_d["text"].sample(n=min(FIT_PER_DOMAIN, len(fit_d)), random_state=42))
    for df in (val, test):
        part = df.index[df["domain_lang"] == dl]
        if len(part) < 2:
            continue
        sim, pos = nearest_other(vec.transform(df.loc[part, "text"]))
        df.loc[part, "nn_sim"] = sim
        df.loc[part, "nn_pos"] = part.to_numpy()[pos]   # row position of the partner in the same frame
print(f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
# Share of texts with a near copy in the same set: validation vs test, per domain and language
def link_share(df):
    return df.assign(linked=df["nn_sim"] >= LINK).groupby("domain_lang").agg(
        texts=("linked", "size"), with_near_copy=("linked", "mean"), median_sim=("nn_sim", "median"))

cmp = link_share(val).join(link_share(test), lsuffix=" val", rsuffix=" test")
cmp.loc["all"] = [len(val), (val["nn_sim"] >= LINK).mean(), val["nn_sim"].median(),
                  len(test), (test["nn_sim"] >= LINK).mean(), test["nn_sim"].median()]
cmp.round(3)

In [ ]:
# Validation: label of each text vs the label of its nearest partner (only texts with a near copy), row shares
linked = val[val["nn_sim"] >= LINK].copy()
linked["partner_label"] = val["label"].to_numpy()[linked["nn_pos"]]
print(f"texts with a near copy: {len(linked)} of {len(val)}; by label:",
      (val["nn_sim"] >= LINK).groupby(val["label"]).mean().round(3).reindex(LABELS).to_dict())
pd.crosstab(linked["label"], linked["partner_label"], normalize="index").reindex(index=LABELS, columns=LABELS).round(3)

In [ ]:
# Human-fusion pairs in validation: does the higher XLM-R human probability point at the human text?
p_human = pd.read_csv(P.runs / "encoder_xlmr" / "predictions" / "val.csv").set_index("id").loc[val["id"], "s_human"].to_numpy()
pairs = linked[linked["label"].isin(["human", "fusion"]) & linked["partner_label"].isin(["human", "fusion"])
               & (linked["label"] != linked["partner_label"])]
a, b = np.unique(np.sort(np.c_[pairs.index.to_numpy(), pairs["nn_pos"].to_numpy()], axis=1), axis=0).T   # each pair once
human_is_a = (val.loc[a, "label"] == "human").to_numpy()
pick_a = p_human[a] > p_human[b]
single = val["label"].isin(["human", "fusion"])
single_acc = ((p_human[single] > 0.5) == (val.loc[single, "label"] == "human")).mean()
print(f"human-fusion near-copy pairs: {len(a)}")
print(f"pair rule (the text with the higher human probability is the human one): {(pick_a == human_is_a).mean():.3f}")
print(f"single texts (human probability > 0.5 means human), human and fusion only: {single_acc:.3f}")

In [ ]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)